In [2]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy"
}

for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])

print("✓ Required libraries are ready!")

import pandas as pd
import numpy as np
import os
import webbrowser
from pathlib import Path

CSV_FILE = r"C:\Users\Nasar\Documents\DATAANALYSIS\hr_analysis_100_records.csv"
OUTPUT_HTML = r"C:\Users\Nasar\Documents\DATAANALYSIS\hr_analytics_dashboard.html"

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"CSV file was not found:\n{CSV_FILE}")

df = pd.read_csv(CSV_FILE)

print("✓ Dataset loaded successfully!")
print(f"✓ Rows: {df.shape[0]}")
print(f"✓ Columns: {df.shape[1]}")

df = df.copy()

df.dropna(axis=0, how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)

object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({
        "nan": np.nan,
        "None": np.nan,
        "": np.nan
    })

df.drop_duplicates(inplace=True)

numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

object_columns = df.select_dtypes(include="object").columns

for col in object_columns:
    if df[col].isna().any():
        mode = df[col].mode()
        fill_value = mode.iloc[0] if not mode.empty else "Unknown"
        df[col] = df[col].fillna(fill_value)

print("✓ Data cleaned successfully!")

data_json = df.to_json(orient="records")

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>HR Analytics Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

<style>
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: "Segoe UI", Arial, sans-serif;
    background: #F4F7FB;
    color: #1F2937;
}

.header {
    background: linear-gradient(135deg, #182848, #2B5876);
    color: white;
    padding: 28px 40px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.12);
}

.header-content {
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.title-section {
    display: flex;
    align-items: center;
    gap: 18px;
}

.logo {
    width: 58px;
    height: 58px;
    background: rgba(255,255,255,0.15);
    border-radius: 16px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 30px;
}

.header h1 {
    font-size: 28px;
}

.header p {
    margin-top: 5px;
    color: #DCE6F2;
}

.header-badge {
    background: rgba(255,255,255,0.15);
    padding: 10px 16px;
    border-radius: 25px;
    font-size: 14px;
}

.container {
    max-width: 1500px;
    margin: auto;
    padding: 30px;
}

.filter-panel {
    background: white;
    padding: 22px;
    border-radius: 18px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.07);
    margin-bottom: 28px;
}

.filter-title {
    font-size: 18px;
    font-weight: 700;
    margin-bottom: 18px;
}

.filters {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
    gap: 15px;
}

.filter-group label {
    display: block;
    font-size: 13px;
    font-weight: 600;
    margin-bottom: 7px;
    color: #4B5563;
}

select {
    width: 100%;
    padding: 11px 12px;
    border: 1px solid #D1D5DB;
    border-radius: 10px;
    background: #FAFAFA;
    font-size: 14px;
    cursor: pointer;
}

select:focus {
    outline: none;
    border-color: #2B5876;
}

.reset-btn {
    background: #2B5876;
    color: white;
    border: none;
    padding: 12px 20px;
    border-radius: 10px;
    cursor: pointer;
    font-size: 14px;
    font-weight: 600;
    align-self: end;
}

.reset-btn:hover {
    background: #182848;
}

.kpi-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 20px;
    margin-bottom: 28px;
}

.kpi-card {
    background: white;
    padding: 22px;
    border-radius: 18px;
    box-shadow: 0 5px 18px rgba(0,0,0,0.07);
    display: flex;
    justify-content: space-between;
    align-items: center;
    transition: 0.25s;
}

.kpi-card:hover {
    transform: translateY(-4px);
    box-shadow: 0 10px 25px rgba(0,0,0,0.12);
}

.kpi-text h3 {
    font-size: 13px;
    color: #6B7280;
    margin-bottom: 9px;
}

.kpi-text h2 {
    font-size: 25px;
    color: #1F2937;
}

.kpi-icon {
    width: 55px;
    height: 55px;
    border-radius: 15px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 26px;
}

.icon-blue {
    background: #E6F0FF;
}

.icon-red {
    background: #FDEBEC;
}

.icon-green {
    background: #E8F7EE;
}

.icon-purple {
    background: #F1EBFF;
}

.chart-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 22px;
    margin-bottom: 25px;
}

.chart-card {
    background: white;
    padding: 22px;
    border-radius: 18px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.07);
}

.chart-card h3 {
    font-size: 17px;
    margin-bottom: 20px;
}

.chart-container {
    position: relative;
    height: 330px;
}

.full-chart {
    grid-column: span 2;
}

.table-card {
    background: white;
    padding: 22px;
    border-radius: 18px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.07);
    margin-top: 25px;
}

.table-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 18px;
}

.table-wrapper {
    overflow-x: auto;
}

table {
    width: 100%;
    border-collapse: collapse;
    min-width: 900px;
}

th {
    background: #F3F6FA;
    color: #4B5563;
    text-align: left;
    padding: 13px;
    font-size: 13px;
}

td {
    padding: 13px;
    border-bottom: 1px solid #EEF0F3;
    font-size: 13px;
}

tr:hover {
    background: #FAFBFD;
}

.badge {
    padding: 5px 10px;
    border-radius: 15px;
    font-size: 12px;
    font-weight: 600;
}

.attrition-yes {
    background: #FDEBEC;
    color: #C0392B;
}

.attrition-no {
    background: #E8F7EE;
    color: #1E8449;
}

.footer {
    text-align: center;
    color: #6B7280;
    padding: 30px 0 10px;
    font-size: 13px;
}

@media(max-width: 900px) {
    .chart-grid {
        grid-template-columns: 1fr;
    }

    .full-chart {
        grid-column: span 1;
    }

    .header-content {
        flex-direction: column;
        align-items: flex-start;
        gap: 15px;
    }
}
</style>
</head>

<body>

<div class="header">
    <div class="header-content">
        <div class="title-section">
            <div class="logo">👥</div>
            <div>
                <h1>HR Analytics Dashboard</h1>
                <p>Workforce Insights • Employee Performance • Attrition Analysis</p>
            </div>
        </div>

        <div class="header-badge">
            📊 Interactive HR Insights
        </div>
    </div>
</div>

<div class="container">

<div class="filter-panel">
    <div class="filter-title">🔎 Filter Employee Data</div>

    <div class="filters">

        <div class="filter-group">
            <label>🏢 Department</label>
            <select id="departmentFilter">
                <option value="All">All Departments</option>
            </select>
        </div>

        <div class="filter-group">
            <label>💼 Job Role</label>
            <select id="roleFilter">
                <option value="All">All Job Roles</option>
            </select>
        </div>

        <div class="filter-group">
            <label>👤 Gender</label>
            <select id="genderFilter">
                <option value="All">All Genders</option>
            </select>
        </div>

        <div class="filter-group">
            <label>🚪 Attrition</label>
            <select id="attritionFilter">
                <option value="All">All Employees</option>
                <option value="Yes">Attrition: Yes</option>
                <option value="No">Attrition: No</option>
            </select>
        </div>

        <div class="filter-group">
            <label>⏰ Overtime</label>
            <select id="overtimeFilter">
                <option value="All">All</option>
            </select>
        </div>

        <button class="reset-btn" onclick="resetFilters()">
            🔄 Reset Filters
        </button>

    </div>
</div>

<div class="kpi-grid">

    <div class="kpi-card">
        <div class="kpi-text">
            <h3>👥 TOTAL EMPLOYEES</h3>
            <h2 id="totalEmployees">0</h2>
        </div>
        <div class="kpi-icon icon-blue">👥</div>
    </div>

    <div class="kpi-card">
        <div class="kpi-text">
            <h3>🚪 ATTRITION RATE</h3>
            <h2 id="attritionRate">0%</h2>
        </div>
        <div class="kpi-icon icon-red">📉</div>
    </div>

    <div class="kpi-card">
        <div class="kpi-text">
            <h3>💰 AVG MONTHLY INCOME</h3>
            <h2 id="avgIncome">0</h2>
        </div>
        <div class="kpi-icon icon-green">💰</div>
    </div>

    <div class="kpi-card">
        <div class="kpi-text">
            <h3>⭐ AVG JOB SATISFACTION</h3>
            <h2 id="avgSatisfaction">0</h2>
        </div>
        <div class="kpi-icon icon-purple">⭐</div>
    </div>

</div>

<div class="chart-grid">

    <div class="chart-card">
        <h3>🏢 Employees by Department</h3>
        <div class="chart-container">
            <canvas id="departmentChart"></canvas>
        </div>
    </div>

    <div class="chart-card">
        <h3>🚪 Attrition Overview</h3>
        <div class="chart-container">
            <canvas id="attritionChart"></canvas>
        </div>
    </div>

    <div class="chart-card">
        <h3>👤 Workforce by Gender</h3>
        <div class="chart-container">
            <canvas id="genderChart"></canvas>
        </div>
    </div>

    <div class="chart-card">
        <h3>💼 Average Income by Department</h3>
        <div class="chart-container">
            <canvas id="incomeChart"></canvas>
        </div>
    </div>

    <div class="chart-card full-chart">
        <h3>⭐ Job Satisfaction by Department</h3>
        <div class="chart-container">
            <canvas id="satisfactionChart"></canvas>
        </div>
    </div>

    <div class="chart-card full-chart">
        <h3>📊 Employees by Job Role</h3>
        <div class="chart-container">
            <canvas id="jobRoleChart"></canvas>
        </div>
    </div>

    <div class="chart-card">
        <h3>⏰ Overtime Distribution</h3>
        <div class="chart-container">
            <canvas id="overtimeChart"></canvas>
        </div>
    </div>

    <div class="chart-card">
        <h3>📈 Age Distribution</h3>
        <div class="chart-container">
            <canvas id="ageChart"></canvas>
        </div>
    </div>

</div>

<div class="table-card">

    <div class="table-header">
        <div>
            <h3>🧾 Employee Records</h3>
            <p style="color:#6B7280; font-size:13px; margin-top:5px;">
                Detailed workforce information
            </p>
        </div>

        <div id="recordCount"
             style="background:#F3F6FA; padding:8px 14px; border-radius:20px; font-size:13px;">
        </div>
    </div>

    <div class="table-wrapper">
        <table>

            <thead>
                <tr>
                    <th>Employee ID</th>
                    <th>Age</th>
                    <th>Gender</th>
                    <th>Department</th>
                    <th>Job Role</th>
                    <th>Monthly Income</th>
                    <th>Years at Company</th>
                    <th>Job Satisfaction</th>
                    <th>Overtime</th>
                    <th>Attrition</th>
                </tr>
            </thead>

            <tbody id="employeeTableBody"></tbody>

        </table>
    </div>
</div>

<div class="footer">
    👥 HR Analytics Dashboard | Built with Python, Pandas, HTML, CSS & Chart.js
</div>

</div>

<script>

const originalData = __DATA_JSON__;

let charts = [];

function getUniqueValues(data, column) {
    return [...new Set(
        data
            .map(item => item[column])
            .filter(value => value !== null && value !== undefined)
    )].sort();
}

function populateFilter(id, column) {

    const select = document.getElementById(id);

    const values = getUniqueValues(
        originalData,
        column
    );

    values.forEach(value => {

        const option = document.createElement("option");

        option.value = value;
        option.textContent = value;

        select.appendChild(option);

    });
}

populateFilter("departmentFilter", "Department");
populateFilter("roleFilter", "Job_Role");
populateFilter("genderFilter", "Gender");
populateFilter("overtimeFilter", "OverTime");

function getFilteredData() {

    const department =
        document.getElementById("departmentFilter").value;

    const role =
        document.getElementById("roleFilter").value;

    const gender =
        document.getElementById("genderFilter").value;

    const attrition =
        document.getElementById("attritionFilter").value;

    const overtime =
        document.getElementById("overtimeFilter").value;

    return originalData.filter(item => {

        return (
            (department === "All" || item.Department === department) &&
            (role === "All" || item.Job_Role === role) &&
            (gender === "All" || item.Gender === gender) &&
            (attrition === "All" || item.Attrition === attrition) &&
            (overtime === "All" || item.OverTime === overtime)
        );

    });
}

function destroyCharts() {

    charts.forEach(chart => chart.destroy());

    charts = [];

}

function countValues(data, column) {

    const result = {};

    data.forEach(item => {

        const value = item[column];

        result[value] = (result[value] || 0) + 1;

    });

    return result;

}

function averageByCategory(data, category, valueColumn) {

    const groups = {};

    data.forEach(item => {

        const key = item[category];

        if (!groups[key]) {
            groups[key] = [];
        }

        groups[key].push(
            Number(item[valueColumn]) || 0
        );

    });

    const result = {};

    Object.keys(groups).forEach(key => {

        const values = groups[key];

        result[key] =
            values.reduce((a, b) => a + b, 0) /
            values.length;

    });

    return result;

}

function updateKPIs(data) {

    const totalEmployees = data.length;

    const attritionEmployees =
        data.filter(item => item.Attrition === "Yes").length;

    const attritionRate =
        totalEmployees > 0
        ? (attritionEmployees / totalEmployees) * 100
        : 0;

    const avgIncome =
        totalEmployees > 0
        ? data.reduce(
            (sum, item) =>
                sum + (Number(item.Monthly_Income) || 0),
            0
          ) / totalEmployees
        : 0;

    const avgSatisfaction =
        totalEmployees > 0
        ? data.reduce(
            (sum, item) =>
                sum + (Number(item.Job_Satisfaction) || 0),
            0
          ) / totalEmployees
        : 0;

    document.getElementById("totalEmployees").textContent =
        totalEmployees.toLocaleString();

    document.getElementById("attritionRate").textContent =
        attritionRate.toFixed(1) + "%";

    document.getElementById("avgIncome").textContent =
        "PKR " + Math.round(avgIncome).toLocaleString();

    document.getElementById("avgSatisfaction").textContent =
        avgSatisfaction.toFixed(2) + " / 4";

}

function createCharts(data) {

    destroyCharts();

    const departmentData =
        countValues(data, "Department");

    charts.push(
        new Chart(
            document.getElementById("departmentChart"),
            {
                type: "bar",
                data: {
                    labels: Object.keys(departmentData),
                    datasets: [{
                        label: "Employees",
                        data: Object.values(departmentData),
                        backgroundColor: "#2B5876",
                        borderRadius: 8
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            display: false
                        }
                    }
                }
            }
        )
    );

    const attritionData =
        countValues(data, "Attrition");

    charts.push(
        new Chart(
            document.getElementById("attritionChart"),
            {
                type: "doughnut",
                data: {
                    labels: Object.keys(attritionData),
                    datasets: [{
                        data: Object.values(attritionData),
                        backgroundColor: [
                            "#D64545",
                            "#2E8B57"
                        ]
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            position: "bottom"
                        }
                    }
                }
            }
        )
    );

    const genderData =
        countValues(data, "Gender");

    charts.push(
        new Chart(
            document.getElementById("genderChart"),
            {
                type: "pie",
                data: {
                    labels: Object.keys(genderData),
                    datasets: [{
                        data: Object.values(genderData),
                        backgroundColor: [
                            "#6C63FF",
                            "#FF8FAB"
                        ]
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            position: "bottom"
                        }
                    }
                }
            }
        )
    );

    const incomeData =
        averageByCategory(
            data,
            "Department",
            "Monthly_Income"
        );

    charts.push(
        new Chart(
            document.getElementById("incomeChart"),
            {
                type: "bar",
                data: {
                    labels: Object.keys(incomeData),
                    datasets: [{
                        label: "Average Income",
                        data: Object.values(incomeData),
                        backgroundColor: "#27AE60",
                        borderRadius: 8
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            display: false
                        }
                    }
                }
            }
        )
    );

    const satisfactionData =
        averageByCategory(
            data,
            "Department",
            "Job_Satisfaction"
        );

    charts.push(
        new Chart(
            document.getElementById("satisfactionChart"),
            {
                type: "bar",
                data: {
                    labels: Object.keys(satisfactionData),
                    datasets: [{
                        label: "Job Satisfaction",
                        data: Object.values(satisfactionData),
                        backgroundColor: "#8E44AD",
                        borderRadius: 8
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        y: {
                            beginAtZero: true,
                            max: 4
                        }
                    },
                    plugins: {
                        legend: {
                            display: false
                        }
                    }
                }
            }
        )
    );

    const roleData =
        countValues(data, "Job_Role");

    charts.push(
        new Chart(
            document.getElementById("jobRoleChart"),
            {
                type: "bar",
                data: {
                    labels: Object.keys(roleData),
                    datasets: [{
                        label: "Employees",
                        data: Object.values(roleData),
                        backgroundColor: "#F39C12",
                        borderRadius: 8
                    }]
                },
                options: {
                    indexAxis: "y",
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            display: false
                        }
                    }
                }
            }
        )
    );

    const overtimeData =
        countValues(data, "OverTime");

    charts.push(
        new Chart(
            document.getElementById("overtimeChart"),
            {
                type: "doughnut",
                data: {
                    labels: Object.keys(overtimeData),
                    datasets: [{
                        data: Object.values(overtimeData),
                        backgroundColor: [
                            "#3498DB",
                            "#E67E22"
                        ]
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            position: "bottom"
                        }
                    }
                }
            }
        )
    );

    const ageGroups = {
        "20-29": 0,
        "30-39": 0,
        "40-49": 0,
        "50+": 0
    };

    data.forEach(item => {

        const age = Number(item.Age);

        if (age >= 20 && age <= 29) {
            ageGroups["20-29"]++;
        }
        else if (age >= 30 && age <= 39) {
            ageGroups["30-39"]++;
        }
        else if (age >= 40 && age <= 49) {
            ageGroups["40-49"]++;
        }
        else if (age >= 50) {
            ageGroups["50+"]++;
        }

    });

    charts.push(
        new Chart(
            document.getElementById("ageChart"),
            {
                type: "bar",
                data: {
                    labels: Object.keys(ageGroups),
                    datasets: [{
                        label: "Employees",
                        data: Object.values(ageGroups),
                        backgroundColor: "#16A085",
                        borderRadius: 8
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            display: false
                        }
                    }
                }
            }
        )
    );

}

function updateTable(data) {

    const tbody =
        document.getElementById("employeeTableBody");

    tbody.innerHTML = "";

    data.forEach(item => {

        const row =
            document.createElement("tr");

        const attritionClass =
            item.Attrition === "Yes"
            ? "attrition-yes"
            : "attrition-no";

        row.innerHTML =
            "<td>" + (item.Employee_ID || "-") + "</td>" +
            "<td>" + (item.Age || "-") + "</td>" +
            "<td>" + (item.Gender || "-") + "</td>" +
            "<td>" + (item.Department || "-") + "</td>" +
            "<td>" + (item.Job_Role || "-") + "</td>" +
            "<td>PKR " +
            Number(item.Monthly_Income || 0).toLocaleString() +
            "</td>" +
            "<td>" + (item.Years_At_Company || 0) + "</td>" +
            "<td>⭐ " + (item.Job_Satisfaction || "-") + "</td>" +
            "<td>" + (item.OverTime || "-") + "</td>" +
            "<td><span class='badge " +
            attritionClass +
            "'>" +
            (item.Attrition || "-") +
            "</span></td>";

        tbody.appendChild(row);

    });

    document.getElementById("recordCount").textContent =
        data.length + " Employee Records";

}

function updateDashboard() {

    const filteredData =
        getFilteredData();

    updateKPIs(filteredData);
    createCharts(filteredData);
    updateTable(filteredData);

}

document.getElementById("departmentFilter")
    .addEventListener("change", updateDashboard);

document.getElementById("roleFilter")
    .addEventListener("change", updateDashboard);

document.getElementById("genderFilter")
    .addEventListener("change", updateDashboard);

document.getElementById("attritionFilter")
    .addEventListener("change", updateDashboard);

document.getElementById("overtimeFilter")
    .addEventListener("change", updateDashboard);

function resetFilters() {

    document.getElementById("departmentFilter").value = "All";
    document.getElementById("roleFilter").value = "All";
    document.getElementById("genderFilter").value = "All";
    document.getElementById("attritionFilter").value = "All";
    document.getElementById("overtimeFilter").value = "All";

    updateDashboard();

}

updateDashboard();

</script>

</body>
</html>
"""

html_content = html_content.replace(
    "__DATA_JSON__",
    data_json
)

with open(
    OUTPUT_HTML,
    "w",
    encoding="utf-8"
) as file:
    file.write(html_content)

print("\n" + "=" * 60)
print("HR INTERACTIVE DASHBOARD CREATED SUCCESSFULLY!")
print("=" * 60)
print(f"\nDashboard saved at:\n{OUTPUT_HTML}")

webbrowser.open_new_tab(
    Path(OUTPUT_HTML).resolve().as_uri()
)

print("\n✓ Dashboard opened in your browser!")

✓ Required libraries are ready!
✓ Dataset loaded successfully!
✓ Rows: 100
✓ Columns: 20
✓ Data cleaned successfully!

HR INTERACTIVE DASHBOARD CREATED SUCCESSFULLY!

Dashboard saved at:
C:\Users\Nasar\Documents\DATAANALYSIS\hr_analytics_dashboard.html

✓ Dashboard opened in your browser!


C:\Users\Nasar\AppData\Local\Temp\ipykernel_2056\2272915971.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns
C:\Users\Nasar\AppData\Local\Temp\ipykernel_2056\2272915971.py:63: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_g